In [ ]:
!pip install openai chromadb pypdf tiktoken python-dotenv

In [ ]:
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader

# -------------------------------
# Setup
# -------------------------------
import os

os.environ["OPENAI_API_KEY"] = ""
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# -------------------------------
# Step 1: Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

# -------------------------------
# Step 2: Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Step 3: Create Embeddings
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Step 4: Store in ChromaDB
# -------------------------------
def create_vector_db(chunks):
    chroma_client = chromadb.Client()
    collection = chroma_client.create_collection(name="rag_demo")

    for i, chunk in enumerate(chunks):
        embedding = get_embedding(chunk)
        collection.add(
            ids=[str(i)],
            embeddings=[embedding],
            documents=[chunk]
        )

    return collection

# -------------------------------
# Step 5: Retrieve Relevant Docs
# -------------------------------
def retrieve(query, collection, top_k=3):
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]
        # Print retrieved chunks
    print("\nRetrieved Chunks:\n" + "-"*50)
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n{doc[:300]}...")  # show first 300 chars

    return docs

# -------------------------------
# Step 6: Generate Answer
# -------------------------------
def generate_answer(query, context_docs):
    context = "\n\n".join(context_docs)

    prompt = f"""
You are a helpful assistant.
Answer the question based ONLY on the context below.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content



In [ ]:
# -------------------------------
# MAIN PIPELINE
# -------------------------------
if __name__ == "__main__":
    pdf_path = "HR Policy Manual 2025.pdf"

    print("📄 Loading PDF...")
    text = load_pdf(pdf_path)

    print("✂️ Chunking...")
    chunks = chunk_text(text)

    print("🧠 Creating Vector DB...")
    collection = create_vector_db(chunks)

    print("✅ Ready for questions!")



In [ ]:
if __name__ == "__main__":
    while True:
          query = input("What is leave policy? ")
          if query.lower() == "exit":
              break

          docs = retrieve(query, collection)
          answer = generate_answer(query, docs)

          print("\n💡 Answer:")
          print(answer)

In [ ]:
!pip install google

In [ ]:
# ================================
# SIMPLE RAG IN ONE CELL (COLAB)
# ================================

# Install dependencies
# !pip install -q openai chromadb pypdf tiktoken

# -------------------------------
# Setup
# -------------------------------
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader
from google.colab import files

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

# -------------------------------
# Upload PDF
# -------------------------------
print("Upload your PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# -------------------------------
# Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

# -------------------------------
# Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Embedding
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Create Vector DB
# -------------------------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

# -------------------------------
# Build Index
# -------------------------------
print("Loading PDF...")
text = load_pdf(pdf_path)

print("Chunking...")
chunks = chunk_text(text)

print("Creating embeddings and storing...")
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk]
    )

print("RAG Ready")

# -------------------------------
# Retriever
# -------------------------------
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    return results["documents"][0]

# -------------------------------
# Generator (LLM)
# -------------------------------
def generate_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

# -------------------------------
# Ask Questions Loop
# -------------------------------
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    ans = generate_answer(q, docs)

    print("Answer:")
    print(ans)

In [ ]:
# ================================
# SIMPLE RAG WITH STREAMING (COLAB)
# ================================

# Install dependencies
!pip install -q openai chromadb pypdf tiktoken

# -------------------------------
# Setup
# -------------------------------
import os
from openai import OpenAI
import chromadb
from pypdf import PdfReader
from google.colab import files

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"
client = OpenAI()

# -------------------------------
# Upload PDF
# -------------------------------
print("Upload your PDF")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# -------------------------------
# Load PDF
# -------------------------------
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

# -------------------------------
# Chunk Text
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# -------------------------------
# Embedding
# -------------------------------
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# -------------------------------
# Create Vector DB
# -------------------------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_demo")

# -------------------------------
# Build Index
# -------------------------------
print("Loading PDF...")
text = load_pdf(pdf_path)

print("Chunking...")
chunks = chunk_text(text)

print("Creating embeddings and storing...")
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[emb],
        documents=[chunk]
    )

print("RAG Ready")

# -------------------------------
# Retriever
# -------------------------------
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    return results["documents"][0]

# -------------------------------
# Streaming Generator
# -------------------------------
def stream_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        stream=True
    )

    full_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            print(token, end="", flush=True)
            full_text += token

    print()  # newline after streaming
    return full_text

# -------------------------------
# Ask Questions Loop
# -------------------------------
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    print("Answer:")
    stream_answer(q, docs)